# 03 · Label & respiratory-data check

Before adding respiratory features, we need to know exactly how the **outcome label** is built, because
the respiratory tables are the same ones that define it (`data/comb/README.md` → *"Modeling Contract"*).

`scripts/build_respiratory_failure_labels.py` marks a stay positive at the **earliest** of, within 0–48 h of ICU admission:

| Criterion | Source | Rule |
|---|---|---|
| A. ventilation | `respiratory_procedureevents.csv` | any procedure whose label contains *intubation / ventilation / endotracheal* |
| B. oxygen escalation | `respiratory_chartevents.csv` | O₂ device value contains *high flow / hfnc / bipap / cpap* |
| C. high FiO₂ | `respiratory_chartevents.csv` | row whose `label` equals `"fio2"` and value ≥ 0.6 |

This notebook checks four things and prints **aggregate counts only**:
1. Which criterion triggers the label, and how often.
2. Whether criterion C can fire at all (label name and units of FiO₂).
3. How many "failures" happen in the first hour (patients who *arrive* ventilated, e.g. after surgery).
4. Which respiratory signals are safe to use as features before a landmark — and which would leak.

Nothing is saved except `label_check_results.json` (aggregate numbers).

In [1]:
import sys, json, re
from pathlib import Path
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# ✏️ Same folders as notebook 02
DATA_DIR = Path("/content/drive/MyDrive/comb")
SAVE_DIR = Path("/content/drive/MyDrive/USC/ICU-MM-landmark")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
LANDMARKS_H = [3, 6, 12]
HORIZON_H = 48
R = {}   # results collected for the JSON

Mounted at /content/drive


In [2]:
# ── Load ──────────────────────────────────────────────────────
cohort = pd.read_csv(DATA_DIR / "cohort.csv", usecols=["stay_id", "icu_intime", "icu_outtime"],
                     parse_dates=["icu_intime", "icu_outtime"])
chart = pd.read_csv(DATA_DIR / "respiratory_chartevents.csv", parse_dates=["charttime"])
proc  = pd.read_csv(DATA_DIR / "respiratory_procedureevents.csv", parse_dates=["starttime", "endtime"])
labels = pd.read_csv(DATA_DIR / "respiratory_failure_labels.csv", parse_dates=["rf_time"])

chart = chart.merge(cohort, on="stay_id", how="inner")
chart["hours"] = (chart["charttime"] - chart["icu_intime"]).dt.total_seconds() / 3600
proc = proc.merge(cohort, on="stay_id", how="inner")
proc["hours"] = (proc["starttime"] - proc["icu_intime"]).dt.total_seconds() / 3600

print(f"chartevents rows: {len(chart):,} | procedureevents rows: {len(proc):,} | label rows: {len(labels):,}")
print(f"Positive rate in label file: {labels['respiratory_failure'].mean()*100:.1f}%")

chartevents rows: 4,761,687 | procedureevents rows: 48,494 | label rows: 94,458
Positive rate in label file: 38.0%


## 1 · What's in the respiratory tables?

In [3]:
print("chartevents — rows per (itemid, label):")
print(chart.groupby(["itemid", "label"]).size().rename("rows").to_string())
print("\nprocedureevents — rows per label:")
print(proc["label"].value_counts().to_string())

dev = chart[chart["itemid"].isin([223834, 226732])]
print("\nO2 device — top 25 values (all stays, all times):")
print(dev["value"].astype(str).str.strip().value_counts().head(25).to_string())
R["chart_rows_by_label"] = chart["label"].value_counts().to_dict()
R["proc_rows_by_label"] = proc["label"].value_counts().to_dict()

chartevents — rows per (itemid, label):
itemid  label                
223834  O2 Flow                   769310
223835  Inspired O2 Fraction     1144289
223848  Ventilator Type           778405
226732  O2 Delivery Device(s)    2069683

procedureevents — rows per label:
label
Invasive Ventilation        35479
Intubation                   9777
Non-invasive Ventilation     3238

O2 device — top 25 values (all stays, all times):
value
Endotracheal tube          633302
Nasal cannula              530952
nan                        398249
2                          218740
Tracheostomy tube          127611
4                          114230
3                           96204
10                          95200
Aerosol-cool                79148
Face tent                   70771
Trach mask                  66480
High flow nasal cannula     48003
15                          42238
5                           39409
6                           37247
12                          28692
Bipap mask            

## 2 · FiO₂: can criterion C fire, and what units are the values in?

In [4]:
fio2 = chart[chart["itemid"] == 223835].copy()
fio2["num"] = pd.to_numeric(fio2["value"], errors="coerce")
label_names = fio2["label"].unique().tolist()
matches_c = (chart["label"].str.lower() == "fio2").sum()
print(f"FiO2 itemid 223835 label name(s) in the data: {label_names}")
print(f"Rows matching the label script's test  label.lower() == 'fio2':  {matches_c:,}")
q = fio2["num"].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
print("\nFiO2 value quantiles:"); print(q.to_string())
share_pct = (fio2["num"] > 1).mean()
print(f"\nShare of FiO2 values > 1 (i.e. recorded as %, not fraction): {share_pct*100:.1f}%")
R["fio2"] = {"label_names": label_names, "rows_matching_label_script": int(matches_c),
             "quantiles": {str(k): float(v) for k, v in q.items()}, "share_values_gt_1": round(float(share_pct), 4)}

FiO2 itemid 223835 label name(s) in the data: ['Inspired O2 Fraction']
Rows matching the label script's test  label.lower() == 'fio2':  0

FiO2 value quantiles:
0.01     28.0
0.05     30.0
0.25     40.0
0.50     40.0
0.75     50.0
0.95    100.0
0.99    100.0

Share of FiO2 values > 1 (i.e. recorded as %, not fraction): 99.9%


## 3 · Which criterion triggers each positive — recomputed exactly like the label script

In [5]:
def first_time(df, mask, col="hours"):
    d = df[mask & (df[col] >= 0) & (df[col] <= HORIZON_H)]
    return d.groupby("stay_id")[col].min()

kw = "high flow|hfnc|bipap|cpap"
dev_all = chart[chart["itemid"].isin([223834, 226732])]
tA = first_time(proc, proc["label"].str.contains("intubation|mechanical ventilation|ventilation|endotracheal", case=False, na=False))
tB = first_time(dev_all, dev_all["value"].astype(str).str.lower().str.contains(kw, na=False))
fio2_in = chart[chart["itemid"] == 223835].copy(); fio2_in["num"] = pd.to_numeric(fio2_in["value"], errors="coerce")
tC_as_written = first_time(fio2_in, (fio2_in["label"].str.lower() == "fio2") & (fio2_in["num"] >= 0.6))
# unit-aware version of criterion C: >= 60 % (or >= 0.6 when recorded as a fraction)
tC_fixed = first_time(fio2_in, ((fio2_in["num"] > 1) & (fio2_in["num"] >= 60)) | ((fio2_in["num"] <= 1) & (fio2_in["num"] >= 0.6)))

T = pd.DataFrame({"A_vent": tA, "B_o2": tB, "C_as_written": tC_as_written}).reindex(cohort["stay_id"])
T["rf_recomputed"] = T.min(axis=1)
lab = labels.set_index("stay_id")
lab["rf_hours"] = (lab["rf_time"] - cohort.set_index("stay_id")["icu_intime"]).dt.total_seconds() / 3600
agree = (T["rf_recomputed"].notna() == (lab["respiratory_failure"].reindex(T.index) == 1)).mean()
print(f"Recomputed label agrees with label file on {agree*100:.2f}% of stays")

pos = T[T["rf_recomputed"].notna()].copy()
pos["first"] = pos[["A_vent", "B_o2", "C_as_written"]].idxmin(axis=1)
print(f"\nPositives: {len(pos):,}. Criterion that fired FIRST:")
print((pos["first"].value_counts(normalize=True) * 100).round(1).astype(str).add("%").to_string())
print("\nStays meeting each criterion at any time in 0–48h:")
for c in ["A_vent", "B_o2", "C_as_written"]:
    print(f"  {c:14s} {T[c].notna().sum():>7,}")
print(f"  C (unit-fixed) {tC_fixed.reindex(cohort['stay_id']).notna().sum():>7,}  ← what criterion C would add if it worked")
R["criteria"] = {"agreement_with_label_file": round(float(agree), 4),
                 "first_trigger_share": pos["first"].value_counts(normalize=True).round(4).to_dict(),
                 "stays_meeting": {c: int(T[c].notna().sum()) for c in ["A_vent", "B_o2", "C_as_written"]},
                 "stays_meeting_C_unit_fixed": int(tC_fixed.reindex(cohort["stay_id"]).notna().sum())}

Recomputed label agrees with label file on 99.99% of stays

Positives: 35,883. Criterion that fired FIRST:
first
A_vent    79.1%
B_o2      20.9%

Stays meeting each criterion at any time in 0–48h:
  A_vent          29,878
  B_o2             9,600
  C_as_written         0
  C (unit-fixed)  29,239  ← what criterion C would add if it worked


## 4 · When do "failures" happen? (arriving ventilated vs. deteriorating later)

In [6]:
rf = lab.loc[lab["respiratory_failure"] == 1, "rf_hours"]
bins = [-np.inf, 0.5, 1, 3, 6, 12, 24, 48]
dist = pd.cut(rf, bins).value_counts(sort=False)
print("Time from ICU admission to label event (positives):")
print((dist / dist.sum() * 100).round(1).astype(str).add("%").to_string())
vent_first_hour = pos[(pos["first"] == "A_vent") & (pos["rf_recomputed"] <= 1)]
print(f"\nPositives triggered by ventilation within the first hour: {len(vent_first_hour):,} "
      f"({len(vent_first_hour)/len(pos)*100:.1f}% of all positives)")
print("These are mostly patients who ARRIVE on a ventilator (e.g. after surgery) — not deterioration in the ICU.")
R["time_to_event_share"] = {str(k): round(float(v), 4) for k, v in (dist / dist.sum()).items()}
R["vent_within_1h_share_of_positives"] = round(len(vent_first_hour) / len(pos), 4)

Time from ICU admission to label event (positives):
rf_hours
(-inf, 0.5]     27.9%
(0.5, 1.0]      10.3%
(1.0, 3.0]      24.0%
(3.0, 6.0]      18.7%
(6.0, 12.0]      8.4%
(12.0, 24.0]     5.9%
(24.0, 48.0]     4.8%

Positives triggered by ventilation within the first hour: 11,450 (31.9% of all positives)
These are mostly patients who ARRIVE on a ventilator (e.g. after surgery) — not deterioration in the ICU.


## 5 · Which pre-landmark respiratory signals are safe features?

For patients still at risk at landmark *L*, look at what was charted **before L** and how often those patients go on to fail.
A signal that is really "already on support" (e.g. an endotracheal tube or a ventilator mode) would make the label almost
certain — that's a leak, not a prediction.

In [7]:
ett_kw = "endotracheal|trach|et tube|ett"
rows = []
for L in LANDMARKS_H:
    st = cohort.set_index("stay_id")
    stay_h = (st["icu_outtime"] - st["icu_intime"]).dt.total_seconds() / 3600
    rfh = lab["rf_hours"].reindex(st.index)
    at_risk = st.index[(stay_h > L) & (rfh.isna() | (rfh > L))]
    y = ((rfh > L) & (rfh <= HORIZON_H)).reindex(at_risk).astype(int)
    pre = chart[(chart["hours"] >= 0) & (chart["hours"] <= L) & chart["stay_id"].isin(at_risk)]
    is_dev = pre["itemid"].isin([223834, 226732])
    pre_val = pre["value"].astype(str).str.lower()
    groups = {
        "any O2 device charted":        pre[is_dev]["stay_id"],
        "ETT / trach device charted":   pre[is_dev & pre_val.str.contains(ett_kw)]["stay_id"],
        "ventilator mode charted":      pre[pre["itemid"] == 223848]["stay_id"],
        "FiO2 charted":                 pre[pre["itemid"] == 223835]["stay_id"],
    }
    base = y.mean()
    for name, ids in groups.items():
        ids = set(ids)
        yin = y[y.index.isin(ids)]
        rows.append({"L": f"{L}h", "signal before L": name, "stays": len(yin),
                     "% of at-risk": round(len(yin) / len(y) * 100, 1),
                     "fail rate with signal": round(yin.mean() * 100, 1) if len(yin) else np.nan,
                     "fail rate overall": round(base * 100, 1)})
sig = pd.DataFrame(rows)
print(sig.to_string(index=False))
R["pre_landmark_signals"] = sig.to_dict(orient="records")

  L            signal before L  stays  % of at-risk  fail rate with signal  fail rate overall
 3h      any O2 device charted  51738          72.2                   12.8               18.9
 3h ETT / trach device charted   6114           8.5                   16.4               18.9
 3h    ventilator mode charted   5753           8.0                   18.9               18.9
 3h               FiO2 charted   9711          13.6                   20.7               18.9
 6h      any O2 device charted  56445          87.7                    9.6               10.7
 6h ETT / trach device charted   7327          11.4                    9.6               10.7
 6h    ventilator mode charted   6917          10.8                   10.2               10.7
 6h               FiO2 charted  11062          17.2                   14.2               10.7
12h      any O2 device charted  55881          94.6                    6.6                6.5
12h ETT / trach device charted   7368          12.5         

In [8]:
with open(SAVE_DIR / "label_check_results.json", "w") as f:
    json.dump(R, f, indent=2, default=str)
print(f"Saved → {SAVE_DIR / 'label_check_results.json'}")

Saved → /content/drive/MyDrive/USC/ICU-MM-landmark/label_check_results.json
